This is the python notebook tutorial introducing ml_client authentication, env creation, and experiment submission. 

## step 1: create workspace handle - ml_client

In [ ]:
# Handle to the workspace
from azure.ai.ml import MLClient

# Authentication package
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()

# Get a handle to the workspace. You can find the info on the workspace tab on ml.azure.com
ml_client = MLClient(
    credential=credential,
    subscription_id="972cf06a-de12-45b1-87f1-3724a2166e79",  # this will look like xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx
    resource_group_name="pme-msan-aml",
    workspace_name="pme-msan-cross-tenant-menghaoyang",
)

In [ ]:
# Verify that the handle works correctly.
# If you ge an error here, modify your SUBSCRIPTION, RESOURCE_GROUP, and WS_NAME in the previous cell.
ws = ml_client.workspaces.get("pme-msan-cross-tenant-menghaoyang")
print(ws.location, ":", ws.resource_group)

eastus : Ads-Singularity-RG-01


## step 2: create custom training enviroment if needed

In [5]:
envs = ml_client.environments.list()
for env in envs:
    print(env.name)

hstu
aml-scikit-learn
IM-vllm-image
HLLM
reform_cuda121_tiantiaw
reform_rocm61_tiantiaw
reform_rocm57_tiantiaw
ngc2410_v6
ngc2410_v5
test
tiantiaw
torch1_11_02
torch1_11_0
sglang-v042post1-rocm620
acpt-stable-ubuntu2004-rocm62-py39-torch221
sglang03
sglang02
sglang01
v-dmuhtar-7a653c52
rocm-flash_attention
CondaEnv-98260459-2906-40ab-b039-734f715fc62b
rocm63_py310
torchtune-rocm
torchtune
nvidia-pytorch
test_image1
llm_distill_cu12_test_v004
hashan_atb
TwinbertEnv
rl_distill-bb13eb74
amlt_project-bb13eb74
PhiMOE
yc_ppo_0903-bb13eb74
yc_ppo-bb13eb74
save_test-735fd63e
abi_rocm
vllm-rocm
rocm_vllm
test_image
bgm-cuda12-1
syomantak-azure
oak_v2-fb15f5dc
llm-inference-sid-aa2a4342
Pytorch-faiss-custom
Pytorch-faiss
syomantak-pytorch-joblib
llm-inferance-qgen-70M-aa2a4342
acpt-rocm61_ubuntu2004_py39_pytorch212
llm_distill_cu12_test_v003
llm_distill_cu12_test_envs
llm_distill_cu12_test
AdsCopilot_reproduce-682a5fdc
sft4ads3-682a5fdc
neel_msan_clf_1-aa2a4342
singularity-pytorch-with-mlflow
te

In [6]:
from azure.ai.ml.entities import Environment

custom_env_name = "aml-scikit-learn"

custom_job_env = Environment(
    name=custom_env_name,
    description="Custom environment for Credit Card Defaults job",
    tags={"scikit-learn": "1.0.2"},
    conda_file="./dependencies/conda.yaml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
)
custom_job_env = ml_client.environments.create_or_update(custom_job_env)

print(
    f"Environment with name {custom_job_env.name} is registered to workspace, the environment version is {custom_job_env.version}"
)

Environment with name aml-scikit-learn is registered to workspace, the environment version is 1


## step3: configure and submit your training job

In [ ]:
from azure.ai.ml import command, Input

registered_model_name = "credit_defaults_model"

# define the command
command_job = command(    
    display_name="credit_default_prediction",
    code='.',
    inputs=dict(
        data=Input(
            type="uri_file",
            path="./data/default_of_credit_card_clients.csv",
        ),
        test_train_ratio=0.2,
        learning_rate=0.25,
        registered_model_name=registered_model_name,
    ),
    command="python main.py --data ${{inputs.data}} --test_train_ratio ${{inputs.test_train_ratio}} --learning_rate ${{inputs.learning_rate}} --registered_model_name ${{inputs.registered_model_name}}",
    environment="aml-scikit-learn@latest",
    compute="/subscriptions/72a1fe05-772c-4836-869f-761a5805fcd4/resourceGroups/Ads-Singularity-RG-01/providers/Microsoft.MachineLearningServices/virtualclusters/ads",
    resources=dict(
        instance_count=1,
        instance_type="Singularity.ND5_v2",
        properties=dict(
            singularity=dict(
                interactive="true",
                imageVersion='',
                slaTier="Premium",
                priority="high",
                tensorboardLogDirectory="/scratch/tensorboard_logs",
                enableAzmlInt="false"
            )
        )

    )
)

In [6]:
# submit the command
returned_job = ml_client.jobs.create_or_update(command_job)
# get a URL for the status of the job
returned_job.studio_url

Uploading getting-started (2.91 MBs): 100%|██████████| 2910182/2910182 [00:01<00:00, 2887895.04it/s]




'https://ml.azure.com/runs/epic_basin_tjmd935411?wsid=/subscriptions/72a1fe05-772c-4836-869f-761a5805fcd4/resourcegroups/Ads-Singularity-RG-01/workspaces/ads-singularity-ws01-eastus&tid=72f988bf-86f1-41af-91ab-2d7cd011db47'